# Predicción de Propensión a Conversión de Servicios


## Contexto del Problema

Este proyecto tiene como objetivo estimar la **probabilidad de conversión** asociada a eventos de recomendación de servicios dentro de un entorno de atención de salud, utilizando información histórica, demográfica y de comportamiento.

El modelo busca anticipar el comportamiento futuro para **priorizar acciones operativas o de contacto**, permitiendo una asignación más eficiente de esfuerzos, sin depender de reglas estáticas ni de incentivos económicos explícitos.



## Dataset y definición de variables

El dataset está compuesto por **datos sintéticos** que simulan eventos asociados a recomendaciones de servicios, incorporando variables de tipo demográfico, clínico (simulado) y de comportamiento histórico.

Se construyeron variables numéricas y categóricas, aplicando técnicas de ingeniería de variables tales como:
- Indexación de variables categóricas
- Escalamiento de variables numéricas
- Ensamblaje de features para modelado


## Modelo de Machine Learning

Se utilizó un modelo **Gradient Boosted Trees (`GBTClassifier`)**, con ajuste de hiperparámetros mediante `CrossValidator`, adecuado para problemas de clasificación binaria con variables mixtas y relaciones no lineales.

Para abordar el desbalance natural entre eventos convertidos y no convertidos, se incorporó una columna de pesos (`weightCol`) que penaliza de forma diferenciada los errores en la clase positiva.

El desempeño del modelo se evaluó utilizando las siguientes métricas:
- Precisión general
- Área bajo la curva ROC
- Área bajo la curva Precision-Recall (PR)
- F1-Score, Precisión y Recall con ajuste de umbral



## Registro del modelo y trazabilidad

El modelo fue registrado utilizando **MLflow**, permitiendo almacenar y versionar los principales artefactos del proceso de entrenamiento para su posterior análisis o reutilización.

En particular, se registraron:
- Hiperparámetros utilizados en el entrenamiento
- Métricas de evaluación del modelo
- Importancia relativa de las variables

Este enfoque asegura **trazabilidad, reproducibilidad y control de versiones** del modelo.


## Ajuste de umbral y normalización de probabilidad

Se identificó un **umbral óptimo** que permite balancear adecuadamente precisión y recall, maximizando el F1-Score del modelo.

A partir de este umbral, la probabilidad estimada fue **normalizada a una escala 0–1**, facilitando su interpretación y uso en etapas posteriores de análisis o segmentación.

Esta probabilidad normalizada puede ser utilizada en un segundo notebook para definir **criterios de priorización o reglas de decisión**, de manera desacoplada del proceso de entrenamiento.



## 1. Extracción de Datos
Descripción de la fuente, filtros aplicados por fecha, y definición del set de entrenamiento y test.

In [ ]:
# ================================
# 📦 Librerías Generales
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tempfile
import pickle
import gc
import sys

# ================================
# 📅 Manejo de Fechas
# ================================
from datetime import date, datetime, timedelta
from dateutil.relativedelta import relativedelta
import dateutil.relativedelta

# ================================
# 📊 Librerías de Modelado (sklearn)
# ================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.metrics import accuracy_score

# ================================
# ⚙️ Configuración de visualización en Pandas
# ================================

# Mostrar los números decimales con 2 cifras
pd.options.display.float_format = '{:.2f}'.format

# Mostrar todas las columnas sin truncar en la salida
pd.set_option('display.max_columns', None)


In [ ]:
# Función para calcular la edad a partir de la fecha de nacimiento
def calcular_edad(df, columna_fecha_nacimiento):
    # Convertir la columna de fecha de nacimiento a tipo datetime si no lo está
    df[columna_fecha_nacimiento] = pd.to_datetime(df[columna_fecha_nacimiento])

    # Obtener la fecha actual
    hoy = pd.to_datetime('today')

    # Calcular la edad fila por fila
    df['edad'] = df[columna_fecha_nacimiento].apply(
        lambda fecha: hoy.year - fecha.year - ((hoy.month, hoy.day) < (fecha.month, fecha.day))
    )

    return df

# Función para obtener la fecha actual con un ajuste de días
def FechaActual(d):
    from datetime import datetime
    import pytz
    zona_horaria = pytz.timezone("America/Santiago")  # Cambiar por la zona horaria que necesites  
    return (datetime.now(zona_horaria) + timedelta(days=d))



### 🗂️ Base de eventos y resultados simulados


In [ ]:
#Definir variable fecha proceso
Fecha_presente = FechaActual(0) 
Fecha_presente

### 🧱 Construcción de la base histórica enriquecida

In [ ]:
# -------------------------------------------------
# Carga de histórico (versión pública)
# -------------------------------------------------
# En un entorno real, esta sección cargaría datos históricos desde almacenamiento distribuido.
# Para fines de portafolio, se simula un histórico consistente con el dataset sintético generado.

try:
    # Simulación: reutilizamos parte del dataset generado previamente
    historical_df = df.copy()
    print("Histórico simulado cargado correctamente")
except Exception as e:
    print(f"Error al cargar histórico simulado: {e}")

# -------------------------------------------------
# Filtros de ejemplo (neutrales)
# -------------------------------------------------
historical_df = historical_df[
    historical_df["service_category_conv"].isin(["Service_A", "Service_B"])
]

historical_df = historical_df[
    historical_df["coverage_type"].isin(["Public_Coverage"])
]

# -------------------------------------------------
# Conversión a Pandas (ya está en Pandas, se mantiene la estructura)
# -------------------------------------------------
historical_df = historical_df.copy()

# Liberar referencia intermedia
del historical_df


In [0]:
# -------------------------------------------------
# Concatenación de histórico simulado + ventana reciente
# -------------------------------------------------
# En versión pública, ambos conjuntos provienen de datos sintéticos

combined_df = pd.concat(
    [historical_df_simulated, bd],
    axis=0,
    ignore_index=True
)

# -------------------------------------------------
# Análisis y normalización de fechas
# -------------------------------------------------
combined_df["event_date"] = pd.to_datetime(combined_df["event_date"])

# Rango temporal del dataset
print("Fecha mínima:", combined_df["event_date"].min())
print("Fecha máxima:", combined_df["event_date"].max())

# -------------------------------------------------
# Eliminación de duplicados
# -------------------------------------------------
combined_df = combined_df.drop_duplicates()

# Reasignar al dataset principal
derivacion_captura = combined_df.copy()

del combined_df



## 2. Transformación y Limpieza

- **Variables categóricas**: transformadas mediante `StringIndexer`, permitiendo su uso en modelos como GBT.
- **Variables numéricas**: ensambladas con `VectorAssembler` y estandarizadas con `StandardScaler` para mejorar el entrenamiento.
- Se construye un `Pipeline` de transformación reutilizable, aplicado tanto al set de entrenamiento como de prueba.


### 🧍 Información del Paciente

### 🔁 Comportamiento Histórico del Paciente

In [0]:
# --- SIMULACIÓN PARA PORTFOLIO: reemplazo de extracción desde fuentes reales ---
import numpy as np
import pandas as pd
from datetime import timedelta

# 1) Fechas
current_date = pd.Timestamp.today().normalize()
end_date = (current_date - timedelta(days=0)).strftime("%Y-%m-%d")       # hoy
start_date = (current_date - timedelta(days=91)).strftime("%Y-%m-%d")   # 91 días atrás

# 2) Generador de datos sintéticos (genérico)
def simulate_service_events(
    start_date: str,
    end_date: str,
    seed: int = 42,
    avg_daily_events: int = 120,  # controla volumen
):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start_date, end=end_date, freq="D")

    # Catálogos (neutralizados)
    service_units = ["Unit_A", "Unit_B", "Unit_C", "Unit_D", "Unit_E"]
    regions = ["Region_Center", "Region_North", "Region_South", "Region_East", "Region_West"]
    service_categories = ["Service_Type_1", "Service_Type_2"]
    business_lines = ["Service_Line"]
    service_groups = [
        "Group_A", "Group_B", "Group_C", "Group_D",
        "Group_E", "Group_F", "Group_G", "Other"
    ]
    coverage_types = ["Public_Coverage"]
    services_a = ["Service_A1", "Service_A2", "Service_A3", "Service_A4"]
    services_b = ["Service_B1", "Service_B2", "Service_B3", "Service_B4"]
    interaction_channels = ["CHANNEL_BOT", "CHANNEL_CALL", "CHANNEL_WEB", "CHANNEL_ONSITE"]

    records = []
    session_counter = 10_000_000

    for d in dates:
        # Variabilidad diaria
        n = max(0, rng.poisson(avg_daily_events))
        if n == 0:
            continue

        # Asignar categorías coherentes
        category = rng.choice(service_categories, size=n, replace=True)
        services = []
        for c in category:
            services.append(rng.choice(services_a if c == "Service_Type_1" else services_b))

        # Probabilidad de conversión e interacción futura
        conversion_flag = rng.binomial(1, 0.38, size=n)
        has_interaction = rng.binomial(1, 0.55, size=n)

        for i in range(n):
            session_counter += 1
            base_price = rng.lognormal(mean=np.log(180000), sigma=0.35)
            price_reference = float(np.clip(base_price, 60_000, 600_000))

            # Evento convertido
            if conversion_flag[i] == 1:
                delta_conv = int(rng.integers(0, 15))
                conversion_date = d + pd.Timedelta(days=delta_conv)
                conversion_unit = rng.choice(service_units)
                session_conv = session_counter
                settled_value = float(np.round(price_reference * rng.uniform(0.85, 1.1), 0))
            else:
                conversion_date = pd.NaT
                conversion_unit = np.nan
                session_conv = np.nan
                settled_value = np.nan

            # Interacción futura (puede existir aunque no convierta)
            if has_interaction[i] == 1:
                delta_inter = int(rng.integers(-2, 31))
                interaction_date = d + pd.Timedelta(days=delta_inter)
                interaction_created = interaction_date - pd.Timedelta(days=int(rng.integers(0, 7)))
                interaction_cancelled = (
                    interaction_date - pd.Timedelta(days=int(rng.integers(0, 3)))
                    if rng.random() < 0.10 else pd.NaT
                )
                interaction_id = int(rng.integers(1_000_000, 9_999_999))
                interaction_channel = rng.choice(interaction_channels)
            else:
                interaction_date = pd.NaT
                interaction_created = pd.NaT
                interaction_cancelled = pd.NaT
                interaction_id = np.nan
                interaction_channel = np.nan

            records.append({
                "event_date": d.normalize(),
                "session_id": session_counter,
                "entity_key": int(rng.integers(5_000_000, 25_000_000)),
                "service_recommended": services[i],
                "service_unit": rng.choice(service_units),
                "region": rng.choice(regions),
                "business_line": rng.choice(business_lines),
                "service_category": category[i],
                "service_group": rng.choice(service_groups),
                "entity_id": int(rng.integers(100_000, 999_999)),
                "coverage_code": "PUB",
                "coverage_type": "Public_Coverage",
                "price_reference": float(np.round(price_reference, 0)),
                "recommendation_flag": 1,
                "conversion": int(conversion_flag[i]),
                "session_recommendation": session_counter,
                "org_unit_code": int(rng.integers(1000, 9999)),
                "service_category_rec": category[i],
                "org_unit_code_conv": int(rng.integers(1000, 9999)) if conversion_flag[i] else np.nan,
                "service_category_conv": category[i] if conversion_flag[i] else np.nan,
                "business_line_conv": "Service_Line" if conversion_flag[i] else np.nan,
                "service_converted": services[i] if conversion_flag[i] else np.nan,
                "conversion_date": conversion_date,
                "session_conversion": session_conv,
                "service_unit_conv": conversion_unit,
                "settled_value": settled_value,
                "interaction_id": interaction_id,
                "interaction_date": interaction_date,
                "interaction_created": interaction_created,
                "interaction_cancelled": interaction_cancelled,
                "interaction_channel": interaction_channel,
                "diagnostic_code": rng.choice(["D01", "D02", "D03", "D04", "D05"]),
                "diagnostic_desc": rng.choice([
                    "General condition",
                    "Follow-up evaluation",
                    "Routine control",
                    "Preventive assessment",
                    "Post-intervention review"
                ]),
                "current_service_number": int(rng.integers(1_000_000, 1_999_999)),
            })

    df = pd.DataFrame(records)

    # Conversión de fechas
    date_cols = [
        "event_date", "conversion_date",
        "interaction_date", "interaction_created", "interaction_cancelled"
    ]
    for c in date_cols:
        df[c] = pd.to_datetime(df[c])

    # Orden final de columnas
    column_order = [
        "event_date","session_id","entity_key","service_recommended","service_unit","region",
        "business_line","service_category","service_group","entity_id","coverage_code",
        "coverage_type","price_reference","recommendation_flag","conversion","session_recommendation",
        "org_unit_code","service_category_rec","org_unit_code_conv",
        "service_category_conv","business_line_conv","service_converted",
        "conversion_date","session_conversion","service_unit_conv","settled_value","interaction_id",
        "interaction_date","interaction_created","interaction_cancelled","interaction_channel",
        "diagnostic_code","diagnostic_desc","current_service_number"
    ]

    return df[column_order].copy()

# 3) “Extracción” simulada
events_df = simulate_service_events(
    start_date=start_date,
    end_date=end_date,
    seed=2025,
    avg_daily_events=110
)

# 4) Limpieza básica
import gc; gc.collect()

# 5) Asegurar tipo fecha
events_df["event_date"] = pd.to_datetime(events_df["event_date"])

# Vista rápida (sin identificadores técnicos)
safe_cols = [c for c in events_df.columns if c not in ["entity_key", "entity_id", "current_service_number"]]
events_df[safe_cols].head(3)


In [0]:
# Limpieza de datos
df = events_df.drop_duplicates().copy()
df["event_date"] = pd.to_datetime(df["event_date"])

# -------------------------------------------------
# Agrupación por sesión / entidad
# -------------------------------------------------
group_session = (
    df.groupby(["entity_id", "business_line_conv", "event_date", "session_id"])
      .agg({
          "service_recommended": "nunique",
          "recommendation_flag": "last",
          "conversion": "sum",
          "conversion_delay": "mean"
      })
      .reset_index()
)

# Ordenar por entidad y fecha (más reciente primero)
group_session = group_session.sort_values(
    ["entity_id", "event_date"], ascending=[True, False]
)

group_session["rank"] = (
    group_session
    .groupby(["entity_id", "business_line_conv"])
    .cumcount() + 1
)

# -------------------------------------------------
# Seleccionar últimas interacciones históricas
# (excluyendo la más reciente)
# -------------------------------------------------
last_interactions = group_session[
    (group_session["rank"] > 1) & (group_session["rank"] <= 4)
].copy()

# -------------------------------------------------
# Features de comportamiento histórico
# -------------------------------------------------
last_interactions["recommendation_non_null"] = (
    last_interactions["recommendation_flag"].replace(0, np.nan)
)

last_interactions["conversion_rate"] = (
    last_interactions["conversion"] /
    last_interactions["recommendation_non_null"]
)

last_interactions["conversion_rate"] = (
    last_interactions["conversion_rate"].fillna(0)
)

last_interactions["conversion_binary"] = (
    last_interactions["conversion"] > 0
).astype(int)

# -------------------------------------------------
# Agregaciones robustas a nivel entidad
# -------------------------------------------------
historical_features = (
    last_interactions
    .groupby(["entity_id", "business_line_conv"])
    .agg({
        "recommendation_flag": ["sum", lambda x: x.std(ddof=0)],
        "conversion": ["sum", lambda x: x.std(ddof=0)],
        "service_recommended": "sum",
        "session_id": "nunique",
        "conversion_delay": ["median", lambda x: x.std(ddof=0)],
        "conversion_rate": ["median", lambda x: x.std(ddof=0)],
        "conversion_binary": "mean"
    })
)

# -------------------------------------------------
# Renombrar columnas (portfolio-safe)
# -------------------------------------------------
historical_features.columns = [
    "num_recommendations", "recommendation_std",
    "num_conversions", "conversion_std",
    "num_services",
    "num_sessions",
    "conversion_delay_median", "conversion_delay_std",
    "conversion_rate_median", "conversion_rate_std",
    "conversion_frequency"
]

historical_features = historical_features.reset_index()

# -------------------------------------------------
# Imputación de valores nulos + flag de historial
# -------------------------------------------------
historical_cols = [
    "num_recommendations", "recommendation_std",
    "num_conversions", "conversion_std",
    "num_services", "num_sessions",
    "conversion_delay_median", "conversion_delay_std",
    "conversion_rate_median", "conversion_rate_std",
    "conversion_frequency"
]

for col in historical_cols:
    historical_features[col] = historical_features[col].fillna(-1)

# Flag binario: ¿existe historial previo?
historical_features["has_history"] = (
    historical_features["num_conversions"] != -1
).astype(int)

# Orden final
historical_features = historical_features.sort_values(
    by="num_services"
)



In [0]:
# Modelo de datos (dataset final para entrenamiento)
base_model = (
    events_df[
        [
            "event_date",
            "session_id",
            "region",
            "business_line_conv",
            "service_category",
            "service_category_conv",
            "entity_id",
            "sex",
            "age",
            "gender_label",
            "coverage_type",
            "year",
            "year_month",
            "month",
            "recommendation_flag",
            "conversion",
            "conversion_delay",
            "outcome_flag",
            "price_reference",
        ]
    ]
    .merge(
        historical_features,
        on=["entity_id", "business_line_conv"],
        how="left"
    )
)


Se definieron dos grandes grupos de variables:

- **Categóricas**:
  - sexo
  - región
  - categoría del servicio
  - tipo de cobertura
  - categoría del servicio convertido

- **Numéricas**:
  - edad
  - volúmenes históricos de recomendaciones y conversiones
  - número de servicios y sesiones previas
  - estadísticas de tiempo a conversión
  - tasas de conversión históricas
  - indicadores de disponibilidad de historial

Estas variables permiten capturar tanto el **perfil de la entidad** como su **comportamiento histórico**, además de la naturaleza del servicio involucrado.


### 🔤 Codificación de Variables Categóricas

In [0]:
# Dataset final para modelado
df = base_model.copy()

# -------------------------------------------------
# Codificación de variables categóricas
# -------------------------------------------------
from sklearn.preprocessing import LabelEncoder

def encode_label(series):
    encoder = LabelEncoder()
    return encoder.fit_transform(series.astype(str))

# Nuevas variables codificadas
df["region_enc"] = encode_label(df["region"])
df["service_category_enc"] = encode_label(df["service_category"])
df["coverage_type_enc"] = encode_label(df["coverage_type"])
df["service_category_conv_enc"] = encode_label(df["service_category_conv"])


### 📌 Selección de Variables Relevantes

In [0]:
# Seleccionar variables
features = ['.......']

df = df[features]

### ⚠️ Manejo de Valores Nulos

In [0]:
# Selección final de variables para modelado
features = [
    # Identificadores mínimos (no usados como feature directa)
    "entity_id",
    "event_date",

    # Demográficas
    "sex",
    "age",

    # Target y variables temporales asociadas
    "conversion",
    "conversion_delay",
    "outcome_flag",

    # Features históricas agregadas
    "num_recommendations",
    "recommendation_std",
    "num_conversions",
    "conversion_std",
    "num_services",
    "num_sessions",
    "conversion_delay_median",
    "conversion_delay_std",
    "conversion_rate_median",
    "conversion_rate_std",
    "conversion_frequency",
    "has_history",

    # Variables categóricas codificadas
    "region_enc",
    "service_category_enc",
    "coverage_type_enc",
    "service_category_conv_enc",
]

df = df[features].copy()



## 4. Entrenamiento del Modelo

Se utilizó un modelo **Gradient-Boosted Trees (GBTClassifier)** por su capacidad de modelar relaciones no lineales y su buen rendimiento en problemas de clasificación binaria con variables mixtas.

- Se aplicó `CrossValidator` con 3 folds y búsqueda en grilla (`maxDepth` y `maxIter`).
- Se incorporó `weightCol` para **aumentar el peso de la clase positiva**, dada la desbalanceada entre pacientes capturados y no capturados.
- Se entrenó y registró el mejor modelo usando **MLflow**.

In [ ]:
# Dataset para entrenamiento del modelo
df_model = df.copy()

# Eliminar duplicados
df_model = df_model.drop_duplicates()

from pyspark.sql import functions as F

# Convertir Pandas DataFrame a Spark DataFrame si es necesario
if isinstance(df_model, pd.DataFrame):
    df_model = spark.createDataFrame(df_model)

# Asegurar el tipo correcto para la columna de fecha
df_model = df_model.withColumn(
    "event_date",
    F.to_date(df_model["event_date"], "yyyy-MM-dd")
)


### ML: GBTClassifier

In [0]:
import mlflow.spark
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array
import matplotlib.pyplot as plt

# -------------------------------------------------
# Crear sesión Spark
# -------------------------------------------------
spark = SparkSession.builder.appName("Propensity_GBT_Model").getOrCreate()

# -------------------------------------------------
# 1. Partición temporal de los datos
# -------------------------------------------------
cutoff_date_train = "2023-01-01"
cutoff_date_start = "2022-01-01"

train = df_model.filter(
    (df_model["event_date"] >= cutoff_date_start) &
    (df_model["event_date"] < cutoff_date_train)
)

test = df_model.filter(df_model["event_date"] >= cutoff_date_train)

# -------------------------------------------------
# 2. Definición de variables
# -------------------------------------------------
categorical_features = [
    "region_enc",
    "service_category_enc",
    "coverage_type_enc",
    "service_category_conv_enc",
]

numerical_features = [
    "age",
    "num_recommendations",
    "recommendation_std",
    "num_conversions",
    "conversion_std",
    "num_services",
    "num_sessions",
    "conversion_delay_median",
    "conversion_delay_std",
    "conversion_rate_median",
    "conversion_rate_std",
    "conversion_frequency",
    "has_history",
    "outcome_flag",
]

label_col = "conversion"

# -------------------------------------------------
# 3. Pipeline de preprocesamiento
# -------------------------------------------------
assembler_num = VectorAssembler(
    inputCols=numerical_features,
    outputCol="num_features_vec"
)

scaler = StandardScaler(
    inputCol="num_features_vec",
    outputCol="scaled_num_features"
)

final_assembler = VectorAssembler(
    inputCols=categorical_features + ["scaled_num_features"],
    outputCol="features"
)

pipeline = Pipeline(stages=[assembler_num, scaler, final_assembler])
pipeline_model = pipeline.fit(train)

train_data = pipeline_model.transform(train).cache()
test_data = pipeline_model.transform(test).cache()

# -------------------------------------------------
# 4. Pesos de clase (manejo de desbalance)
# -------------------------------------------------
train_data = train_data.withColumn(
    "weight",
    when(col(label_col) == 1, 3.0).otherwise(1.0)
)

train_count = train_data.count()
test_count = test_data.count()

# -------------------------------------------------
# 5. Entrenamiento + MLflow
# -------------------------------------------------
with mlflow.start_run(nested=True):

    mlflow.log_params({
        "train_start_date": cutoff_date_start,
        "train_end_date": cutoff_date_train,
        "num_train_samples": train_count,
        "num_test_samples": test_count,
        "model_type": "GBTClassifier",
        "scaler": "StandardScaler",
        "class_weighting": "enabled",
        "positive_class_weight": 3.0,
    })

    gbt = GBTClassifier(
        labelCol=label_col,
        featuresCol="features",
        seed=42,
        maxBins=100,
        weightCol="weight"
    )

    paramGrid = (
        ParamGridBuilder()
        .addGrid(gbt.maxDepth, [4, 6, 8])
        .addGrid(gbt.maxIter, [30, 50])
        .build()
    )

    evaluator_pr = BinaryClassificationEvaluator(
        labelCol=label_col,
        metricName="areaUnderPR"
    )

    crossval = CrossValidator(
        estimator=gbt,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator_pr,
        numFolds=3
    )

    cv_model = crossval.fit(train_data)
    predictions = cv_model.transform(test_data)

    pr_auc = evaluator_pr.evaluate(predictions)
    roc_auc = BinaryClassificationEvaluator(
        labelCol=label_col,
        metricName="areaUnderROC"
    ).evaluate(predictions)

    mlflow.log_metric("areaUnderPR", pr_auc)
    mlflow.log_metric("areaUnderROC", roc_auc)

    print(f"Área bajo curva PR:  {pr_auc:.4f}")
    print(f"Área bajo curva ROC: {roc_auc:.4f}")

# -------------------------------------------------
# 6. Optimización de umbral por F1
# -------------------------------------------------
predictions = predictions.withColumn(
    "prob_array",
    vector_to_array("probability")
)

def find_best_threshold(predictions, label_col, prob_col="prob_array", metric="f1", plot=True):
    thresholds = [x / 100 for x in range(10, 90, 5)]
    scores = []

    evaluator = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction_tmp",
        metricName=metric
    )

    for t in thresholds:
        tmp = predictions.withColumn(
            "prediction_tmp",
            when(col(prob_col)[1] >= t, 1.0).otherwise(0.0)
        )
        scores.append(evaluator.evaluate(tmp))

    best_idx = scores.index(max(scores))
    best_threshold = thresholds[best_idx]

    if plot:
        plt.figure(figsize=(8, 4))
        plt.plot(thresholds, scores, marker="o")
        plt.xlabel("Threshold")
        plt.ylabel(metric.upper())
        plt.title(f"{metric.upper()} vs Threshold")
        plt.grid(True)
        plt.show()

    return best_threshold, scores[best_idx]

best_threshold, best_f1 = find_best_threshold(
    predictions, label_col=label_col, metric="f1", plot=True
)

mlflow.log_metric("best_threshold_f1", best_threshold)
mlflow.log_metric("best_f1", best_f1)

print(f"✅ Mejor umbral F1: {best_threshold:.2f} | F1: {best_f1:.4f}")

# -------------------------------------------------
# 7. Métricas finales con umbral ajustado
# -------------------------------------------------
predictions = predictions.withColumn(
    "prediction_final",
    when(col("prob_array")[1] >= best_threshold, 1.0).otherwise(0.0)
)

print("📊 Métricas con umbral optimizado:")

for metric in ["f1", "precisionByLabel", "recallByLabel"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction_final",
        metricName=metric
    )
    value = evaluator.evaluate(predictions)
    mlflow.log_metric(f"{metric}_optimized", value)
    print(f"{metric}: {value:.4f}")

# -------------------------------------------------
# 8. Matriz de confusión
# -------------------------------------------------
print(f"📊 Matriz de confusión (threshold = {best_threshold})")
predictions.groupBy(label_col, "prediction_final").count().orderBy(
    label_col, "prediction_final"
).show()

# -------------------------------------------------
# 9. Guardado del modelo
# -------------------------------------------------
from pyspark.ml import PipelineModel
import time

final_model = PipelineModel(
    stages=pipeline_model.stages + [cv_model.bestModel]
)

BASE_PATH = "dbfs:/Models/propensity_gbt_model"
VERSION_PATH = f"{BASE_PATH}/v={int(time.time())}"

mlflow.spark.log_model(
    cv_model.bestModel,
    artifact_path="best_model"
)

final_model.write().overwrite().save(BASE_PATH)
final_model.write().overwrite().save(VERSION_PATH)

mlflow.log_param("decision_threshold", float(best_threshold))
mlflow.log_dict(
    {
        "categorical_features": categorical_features,
        "numerical_features": numerical_features,
        "label": label_col,
        "class_weight": 3.0,
        "model_path_latest": BASE_PATH,
        "model_path_versioned": VERSION_PATH,
    },
    "model_metadata.json"
)

print("✅ Modelo guardado correctamente")
print(f"   - Última versión: {BASE_PATH}")
print(f"   - Versión específica: {VERSION_PATH}")


## 5. Evaluación del Modelo

El modelo fue evaluado con las siguientes métricas:

- **Área bajo curva PR (Precision-Recall)**: 0.712
- **Área bajo curva ROC**: 0.769
- Se ajustó el umbral de decisión usando F1-score, resultando en:
  - **Umbral óptimo**: 0.80
  - **F1-score**: 0.761
  - **Precisión clase positiva**: 0.78
  - **Recall clase positiva**: 0.94

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when

# 1. Convertir la probabilidad del modelo a un array accesible
predictions = predictions.withColumn(
    "prob_array",
    vector_to_array("probability")
)

# 2. Definir umbral óptimo (obtenido en la etapa de evaluación)
optimal_threshold = 0.80  # ajustable según criterio de negocio o métrica objetivo

# 3. Crear predicción binaria usando el umbral óptimo
predictions = predictions.withColumn(
    "prediction_final",
    when(col("prob_array")[1] >= optimal_threshold, 1.0).otherwise(0.0)
)

# 4. Normalizar la probabilidad de conversión a escala 0–1
predictions = predictions.withColumn(
    "normalized_probability",
    (col("prob_array")[1] / optimal_threshold).cast("double")
)



## 6. Aplicación y lógica de decisión

Las predicciones del modelo se utilizan como un **score de propensión**, que permite priorizar y segmentar casos según su probabilidad estimada de conversión.

- **Propensión baja**: casos que pueden requerir mayor seguimiento o intervención temprana.
- **Propensión intermedia**: casos donde una gestión o recordatorio oportuno puede mejorar la probabilidad de conversión.
- **Propensión alta**: casos con alta probabilidad de conversión, que requieren una intervención mínima.

Este enfoque permite **optimizar la asignación de esfuerzos operativos**, enfocando la atención donde existe mayor incertidumbre y evitando intervenciones innecesarias en casos con alta probabilidad de éxito.



## 7. Conclusiones

El modelo proporciona una estimación robusta de la **probabilidad de conversión**, permitiendo priorizar casos de forma consistente y basada en datos.

Su desempeño demuestra que es posible **mejorar la eficiencia operativa** mediante una mejor asignación de esfuerzos, enfocando la atención en los casos con mayor incertidumbre y evitando intervenciones innecesarias en aquellos con alta probabilidad de éxito.

Este enfoque es escalable y puede integrarse fácilmente en flujos de análisis, sistemas de monitoreo o procesos de toma de decisiones basados en propensión.
